<a href="https://colab.research.google.com/github/monowar-mukul/AI-102_Azure-AI-Engineer-Associate--Labs/blob/main/healthcare_support_agents_llm_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM Healthcare Support Agents



- One shared `ask_llm()` helper acts as the common LLM brain.
- Each function is one focused agent.
- The pipeline supports appointment, FAQ, triage, handoff, and safety review.
- Deterministic safety checks wrap the LLM so urgent symptoms are escalated safely.

This is an educational demo. It does not provide diagnosis, treatment, or emergency care.


## 1. Install and Configure the LLM Client

This version uses Groq, matching the workshop notebook style. You need a Groq API key.

Do not paste keys into shared notebooks. The cell below asks for the key at runtime.


In [1]:
!pip -q install groq

from getpass import getpass
from groq import Groq
import json
import time
from dataclasses import dataclass
from typing import Iterable

GROQ_API_KEY = getpass("Paste your Groq API key: ")
client = Groq(api_key=GROQ_API_KEY)

MODEL = "llama-3.1-8b-instant"


def ask_llm(prompt: str, system: str = "You are a careful healthcare support assistant.", retries: int = 3) -> str:
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user", "content": prompt},
                ],
                temperature=0.2,
            )
            return response.choices[0].message.content.strip()
        except Exception as exc:
            if attempt < retries - 1 and ("429" in str(exc) or "rate" in str(exc).lower()):
                time.sleep(2 ** attempt)
            else:
                raise


def parse_json_object(raw: str) -> dict:
    cleaned = raw.replace("```json", "").replace("```", "").strip()
    start = cleaned.find("{")
    end = cleaned.rfind("}")
    if start >= 0 and end >= start:
        cleaned = cleaned[start : end + 1]
    return json.loads(cleaned)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.0 MB/s eta 0:00:00
Paste your Groq API key: ··········


## 2. Shared Data and Safety Policy

In [2]:
@dataclass(frozen=True) #create immutable object
class AgentResponse:
    category: str
    answer: str
    escalation: str | None = None
    next_steps: tuple[str, ...] = ()


FAQS = {
    "hours": "The clinic is open Monday-Friday, 8:00 AM-6:00 PM, and Saturday, 9:00 AM-1:00 PM.",
    "insurance": "Bring your insurance card and photo ID. Coverage questions should be confirmed with your insurer.",
    "refills": "Medication refill requests are routed to the care team and are usually reviewed within one business day.",
    "records": "Medical records can be requested through the patient portal or front desk.",
    "telehealth": "Telehealth visits are available for many follow-ups and some routine concerns.",
}

APPOINTMENT_SLOTS = [
    "Monday 9:30 AM with Dr. Rivera",
    "Tuesday 2:00 PM with Nurse Patel",
    "Thursday 11:15 AM telehealth follow-up",
]

EMERGENCY_RED_FLAGS = {
    "chest pain",
    "trouble breathing",
    "shortness of breath",
    "stroke",
    "fainting",
    "severe bleeding",
    "suicidal",
    "overdose",
    "anaphylaxis",
}

CLINICIAN_ESCALATION_TERMS = {
    "pregnant",
    "infant",
    "fever for 3 days",
    "worsening",
    "severe pain",
    "new medication reaction",
    "confusion",
    "dehydration",
}

APPOINTMENT_TERMS = {"appointment", "book", "schedule", "reschedule", "cancel", "visit"}
FAQ_TERMS = {"hours", "insurance", "refill", "records", "telehealth", "portal"}
TRIAGE_TERMS = {"symptom", "pain", "fever", "cough", "rash", "dizzy", "nausea", "breathing", "bleeding", "headache"}


def normalize(text: str) -> str:
    return " ".join(text.lower().strip().split())


def contains_any(text: str, terms: Iterable[str]) -> bool:
    lowered = normalize(text)
    return any(term in lowered for term in terms)


def deterministic_route(message: str) -> str:
    lowered = normalize(message)
    if contains_any(lowered, EMERGENCY_RED_FLAGS | CLINICIAN_ESCALATION_TERMS | TRIAGE_TERMS):
        return "triage"
    if contains_any(lowered, APPOINTMENT_TERMS):
        return "appointment"
    if contains_any(lowered, FAQ_TERMS):
        return "faq"
    return "handoff"


## 3. LLM Function-Agents

In [3]:
def intake_agent(message: str) -> dict:
    safe_route = deterministic_route(message)
    if safe_route == "triage":
        return {"route": "triage", "reason": "Safety terms detected before LLM routing."}

    prompt = f"""
Classify this patient support message into exactly one route:
- appointment: scheduling, rescheduling, canceling, or visit availability
- faq: clinic hours, insurance, refills, records, portal, or telehealth
- triage: symptoms or health concerns
- handoff: anything else or low confidence

Return ONLY JSON:
{{"route": "appointment|faq|triage|handoff", "reason": "short reason"}}

Message: {message}
"""
    try:
        result = parse_json_object(ask_llm(prompt))
        route = result.get("route", "handoff")
        if route not in {"appointment", "faq", "triage", "handoff"}:
            route = "handoff"
        return {"route": route, "reason": result.get("reason", "LLM classified the request.")}
    except Exception as exc:
        return {"route": safe_route, "reason": f"LLM parse failed; used deterministic route. Error: {exc}"}


def appointment_agent(message: str) -> AgentResponse:
    prompt = f"""
You are a clinic scheduling assistant.
Use only these available slots:
{json.dumps(APPOINTMENT_SLOTS, indent=2)}

Write a concise patient-facing response. Do not confirm a booking yet.
Message: {message}
"""
    answer = ask_llm(prompt)
    return AgentResponse(
        category="appointment",
        answer=answer,
        next_steps=("Ask the patient to choose a slot.", "Confirm contact details before booking."),
    )


def faq_agent(message: str) -> AgentResponse:
    prompt = f"""
Answer this clinic FAQ using only the knowledge base below.
If the knowledge base does not contain the answer, say it should be routed to clinic staff.

Knowledge base:
{json.dumps(FAQS, indent=2)}

Return ONLY JSON:
{{"answer": "patient-facing answer", "confidence": "high|low"}}

Message: {message}
"""
    try:
        result = parse_json_object(ask_llm(prompt))
        answer = result.get("answer", "")
        confidence = result.get("confidence", "low")
        if confidence != "high":
            return handoff_agent(message, "FAQ confidence was low.")
        return AgentResponse(category="faq", answer=answer)
    except Exception as exc:
        return handoff_agent(message, f"FAQ agent could not parse the LLM response: {exc}")


def triage_agent(message: str) -> AgentResponse:
    lowered = normalize(message)
    if contains_any(lowered, EMERGENCY_RED_FLAGS):
        return AgentResponse(
            category="triage",
            answer=(
                "This may be urgent. I cannot provide emergency care here. "
                "Please call emergency services or go to the nearest emergency department now."
            ),
            escalation="emergency",
            next_steps=("Call local emergency services.", "Do not wait for an online response."),
        )

    if contains_any(lowered, CLINICIAN_ESCALATION_TERMS):
        prompt = f"""
Write a brief patient-facing response that says a clinician should review this request.
Do not diagnose. Do not give treatment instructions.
Message: {message}
"""
        answer = ask_llm(prompt)
        return AgentResponse(
            category="triage",
            answer=answer,
            escalation="clinician",
            next_steps=("Create clinician callback ticket.", "Share symptom duration and severity."),
        )

    prompt = f"""
Write a brief, safe nurse-triage style response.
Do not diagnose. Do not give treatment instructions.
Recommend routine scheduling and monitoring for urgent or worsening symptoms.
Message: {message}
"""
    answer = ask_llm(prompt)
    return AgentResponse(
        category="triage",
        answer=answer,
        next_steps=("Offer appointment slots.", "Advise monitoring for urgent or worsening symptoms."),
    )


def handoff_agent(message: str, reason: str = "Low confidence or unsupported request.") -> AgentResponse:
    prompt = f"""
Write a concise patient-facing handoff response.
Reason: {reason}
Message: {message}
The response should say clinic staff will review rather than guessing.
"""
    answer = ask_llm(prompt)
    return AgentResponse(
        category="handoff",
        answer=answer,
        escalation="support_staff",
        next_steps=("Collect patient contact information.", "Create support handoff ticket."),
    )


## 4. Safety Review and Pipeline

In [4]:
def safety_review_agent(message: str, response: AgentResponse) -> AgentResponse:
    lowered = normalize(message)
    if contains_any(lowered, EMERGENCY_RED_FLAGS):
        return AgentResponse(
            category="triage",
            answer=(
                "This may be urgent. I cannot provide emergency care here. "
                "Please call emergency services or go to the nearest emergency department now."
            ),
            escalation="emergency",
            next_steps=("Call local emergency services.", "Do not wait for an online response."),
        )

    if response.category != "triage" and contains_any(lowered, TRIAGE_TERMS | CLINICIAN_ESCALATION_TERMS):
        return AgentResponse(
            category="triage",
            answer="A clinician should review this before guidance is given.",
            escalation="clinician",
            next_steps=("Create clinician callback ticket.", "Share symptom duration and severity."),
        )

    if response.category != "triage" and not response.escalation:
        return AgentResponse(
            category=response.category,
            answer=response.answer,
            escalation=response.escalation,
            next_steps=response.next_steps + ("Escalate if symptoms are urgent or worsening.",),
        )

    return response


def run_llm_pipeline(message: str) -> dict:
    trace = []
    intake = intake_agent(message)
    route = intake["route"]
    trace.append(f"Intake Agent -> route={route}; reason={intake['reason']}")

    if route == "appointment":
        draft = appointment_agent(message)
        trace.append("Appointment Agent -> used LLM to draft scheduling response")
    elif route == "faq":
        draft = faq_agent(message)
        trace.append("FAQ Agent -> used LLM with clinic knowledge base")
    elif route == "triage":
        draft = triage_agent(message)
        trace.append("Triage Agent -> used safety policy plus LLM wording where appropriate")
    else:
        draft = handoff_agent(message)
        trace.append("Handoff Agent -> used LLM to draft staff handoff")

    final = safety_review_agent(message, draft)
    trace.append("Safety Review Agent -> deterministic final guardrail applied")

    return {
        "response": {
            "category": final.category,
            "answer": final.answer,
            "escalation": final.escalation,
            "next_steps": list(final.next_steps),
        },
        "trace": trace,
    }


## 5. Run the LLM Agents

In [5]:
examples = [
    "Can I schedule an appointment next week?",
    "What are your clinic hours?",
    "I am pregnant and have worsening severe pain",
    "I have chest pain and trouble breathing",
]

for message in examples:
    print("=" * 80)
    print("Patient message:", message)
    result = run_llm_pipeline(message)
    print(json.dumps(result, indent=2))


Patient message: Can I schedule an appointment next week?
{
  "response": {
    "category": "appointment",
    "answer": "\"Hello, we have a few available slots next week. Would you like to consider the following options: \n- Monday 9:30 AM with Dr. Rivera\n- Tuesday 2:00 PM with Nurse Patel\n- Thursday 11:15 AM telehealth follow-up\n\nPlease let me know if any of these times work for you, and we can proceed with booking an appointment.\"",
    "escalation": null,
    "next_steps": [
      "Ask the patient to choose a slot.",
      "Confirm contact details before booking.",
      "Escalate if symptoms are urgent or worsening."
    ]
  },
  "trace": [
    "Intake Agent -> route=appointment; reason=scheduling inquiry",
    "Appointment Agent -> used LLM to draft scheduling response",
    "Safety Review Agent -> deterministic final guardrail applied"
  ]
}
Patient message: What are your clinic hours?
{
  "response": {
    "category": "faq",
    "answer": "The clinic is open Monday-Friday,

## 6. Agentic Pattern Trace with LLM

In [6]:
def planner_agent(message: str) -> list[str]:
    prompt = f"""
Create a short plan for handling this healthcare support message.
Use safe escalation for symptoms.
Return ONLY JSON:
{{"steps": ["step 1", "step 2", "step 3"]}}
Message: {message}
"""
    try:
        return parse_json_object(ask_llm(prompt)).get("steps", [])
    except Exception:
        return ["Classify request", "Run workflow", "Apply safety review"]


def rewoo_lookup_agent(message: str) -> tuple[dict, list[str]]:
    route = deterministic_route(message)
    trace = [f"Plan lookup first: route={route}"]
    context = {}
    if route == "faq":
        context["knowledge_base"] = FAQS
        trace.append("Lookup executed: clinic FAQ knowledge")
    elif route == "appointment":
        context["available_slots"] = APPOINTMENT_SLOTS
        trace.append("Lookup executed: appointment slots")
    elif route == "triage":
        context["safety_terms"] = {
            "emergency_red_flags": sorted(EMERGENCY_RED_FLAGS),
            "clinician_escalation_terms": sorted(CLINICIAN_ESCALATION_TERMS),
        }
        trace.append("Lookup executed: safety terms")
    return context, trace


def reflection_agent(message: str, pipeline_result: dict) -> tuple[dict, list[str]]:
    prompt = f"""
Review this draft healthcare support response for safety.
Do not rewrite unless it misses escalation or sounds diagnostic.
Return ONLY JSON:
{{"approved": true, "note": "short review note"}}

Message: {message}
Draft response:
{json.dumps(pipeline_result["response"], indent=2)}
"""
    try:
        review = parse_json_object(ask_llm(prompt))
        return pipeline_result, [f"LLM reflection approved={review.get('approved')}: {review.get('note')}"]
    except Exception as exc:
        return pipeline_result, [f"Reflection parse failed; deterministic safety already applied. Error: {exc}"]


def run_agentic_llm_pipeline(message: str) -> dict:
    plan = planner_agent(message)
    lookup_context, rewoo_trace = rewoo_lookup_agent(message)
    result = run_llm_pipeline(message)
    reviewed_result, reflection_trace = reflection_agent(message, result)
    reviewed_result["patterns"] = {
        "planning": plan,
        "rewoo": rewoo_trace,
        "tool_lookup_context": lookup_context,
        "react": result["trace"],
        "reflection": reflection_trace,
        "multi_agent": [
            "Planner Agent made a plan",
            "Lookup Agent gathered knowledge before answering",
            "Workflow Agents drafted the response",
            "Safety Review Agent applied deterministic guardrails",
            "Reflection Agent reviewed the final draft",
        ],
    }
    return reviewed_result


message = "I am pregnant and have worsening severe pain"
print(json.dumps(run_agentic_llm_pipeline(message), indent=2))


{
  "response": {
    "category": "triage",
    "answer": "\"Thank you for reaching out to us about your concerns. I've noted your message and will pass it along to a clinician for review. They will be in touch with you as soon as possible to discuss your symptoms and provide further guidance.\"",
    "escalation": "clinician",
    "next_steps": [
      "Create clinician callback ticket.",
      "Share symptom duration and severity."
    ]
  },
  "trace": [
    "Intake Agent -> route=triage; reason=Safety terms detected before LLM routing.",
    "Triage Agent -> used safety policy plus LLM wording where appropriate",
    "Safety Review Agent -> deterministic final guardrail applied"
  ],
  "patterns": {
    "planning": [
      "Step 1: Assess the severity of the pain and any other symptoms, such as nausea, vomiting, or fever. If the pain is severe and accompanied by any of these symptoms, call emergency services immediately.",
      "Step 2: If the pain is severe but not accompanied by

## 7. Optional Interactive UI

In [7]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    message_box = widgets.Textarea(
        value="Can I schedule an appointment?",
        description="Message:",
        layout=widgets.Layout(width="100%", height="90px"),
    )
    mode = widgets.ToggleButtons(
        options=["Simple LLM pipeline", "Agentic LLM pipeline"],
        description="Mode:",
    )
    run_button = widgets.Button(description="Run Agents", button_style="primary")
    output = widgets.Output()

    def on_run_clicked(_):
        with output:
            clear_output()
            if mode.value == "Agentic LLM pipeline":
                result = run_agentic_llm_pipeline(message_box.value)
            else:
                result = run_llm_pipeline(message_box.value)
            print(json.dumps(result, indent=2))

    run_button.on_click(on_run_clicked)
    display(widgets.VBox([message_box, mode, run_button, output]))
except Exception as exc:
    print("Interactive widgets are unavailable in this runtime:", exc)
